# Blender Resumable Blocks — ChatGPT Work\n\nRender reanudable por bloques persistentes en Google Drive. Si Colab se desconecta, los bloques ya terminados no se vuelven a renderizar.\n\nPor defecto usa `blender-neon-core-demo`, bloques de 12 frames y EEVEE.\n\nUso: abre el cuaderno y pulsa **Ejecutar todo**.\n

In [ ]:
# @title Blender resumable blocks — ejecutar todo
# Render reanudable por bloques con guardado persistente en Drive.
# Por defecto usa Neon Core y EEVEE. Cada bloque terminado queda guardado.

VERSION_MARKER = "RESUMABLE-BLOCKS-V1-2026-10-02"
PROJECT_SLUG = "blender-neon-core-demo"
BLOCK_SIZE = 12

REPO_ZIP = "https://github.com/izquierdodavid528-code/Chat-gpt-work-/archive/refs/heads/main.zip"
REPO_DIR = "/content/chatgpt-workspace"
BLENDER_VERSION = "4.5.14"
BLENDER_DIR = f"/content/blender-{BLENDER_VERSION}-linux-x64"
BLENDER_BIN = f"{BLENDER_DIR}/blender"

import os
os.chdir("/content")

import json
import math
import time
import shutil
import pathlib
import subprocess
import urllib.request
import zipfile
import tarfile
from datetime import datetime, timezone

def section(title):
    print("\n" + "="*78)
    print(title)
    print("="*78)

def run(cmd, check=True, capture=False):
    p = subprocess.run(
        ["bash","-lc",cmd],
        cwd="/content",
        text=True,
        capture_output=capture
    )
    if capture:
        if p.stdout: print(p.stdout)
        if p.stderr: print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Comando fallo ({p.returncode}): {cmd}")
    return p

section(VERSION_MARKER)

section("1/8 - Montando Drive")
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/Remotion Projects")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

section("2/8 - Descargando workspace")
workspace_zip = pathlib.Path("/content/workspace-main.zip")
extract_root = pathlib.Path("/content/workspace-extract")

for p in [pathlib.Path(REPO_DIR), extract_root]:
    if p.exists():
        shutil.rmtree(p)
workspace_zip.unlink(missing_ok=True)

urllib.request.urlretrieve(REPO_ZIP, workspace_zip)
extract_root.mkdir(parents=True)

with zipfile.ZipFile(workspace_zip,"r") as z:
    names=z.namelist()
    if not names:
        raise RuntimeError("ZIP del repositorio vacio.")
    top=names[0].split("/")[0]
    z.extractall(extract_root)

src_root=extract_root/top
shutil.move(str(src_root), REPO_DIR)
shutil.rmtree(extract_root, ignore_errors=True)
workspace_zip.unlink(missing_ok=True)
print("Workspace OK:", REPO_DIR)

section("3/8 - Instalando Blender")
run(
    "apt-get -qq update && "
    "apt-get -qq install -y curl ca-certificates ffmpeg "
    "libegl1 libgl1 libx11-6 libxi6 libxrender1 libxfixes3 "
    "libxxf86vm1 libxkbcommon0"
)

archive=pathlib.Path("/content/blender.tar.xz")
archive.unlink(missing_ok=True)

if not pathlib.Path(BLENDER_BIN).exists():
    filename=f"blender-{BLENDER_VERSION}-linux-x64.tar.xz"
    urls=[
        f"https://download.blender.org/release/Blender4.5/{filename}",
        f"https://ftp.nluug.nl/pub/graphics/blender/release/Blender4.5/{filename}",
    ]

    ok=False
    for i,url in enumerate(urls,1):
        print(f"Intento {i}/{len(urls)}:",url)
        q=subprocess.run(
            [
                "curl","-fL",
                "--retry","4",
                "--retry-delay","2",
                "--connect-timeout","30",
                "--max-time","900",
                "-A","Mozilla/5.0",
                url,
                "-o",str(archive),
            ],
            cwd="/content",
        )
        if q.returncode==0 and archive.exists() and archive.stat().st_size>300_000_000:
            ok=True
            break
        archive.unlink(missing_ok=True)

    if not ok:
        raise RuntimeError("No se pudo descargar Blender.")

    with tarfile.open(archive,"r:xz") as tf:
        tf.extractall("/content")
    archive.unlink(missing_ok=True)

ver=subprocess.run([BLENDER_BIN,"--version"],cwd="/content",text=True,capture_output=True)
if ver.returncode!=0:
    raise RuntimeError("Blender no pudo iniciar.")
print("\n".join(ver.stdout.splitlines()[:3]))

section("4/8 - Preparando proyecto")
project_dir=pathlib.Path(REPO_DIR)/"projects"/PROJECT_SLUG
config_path=project_dir/"project.config.json"
scene_script=project_dir/"scene.py"

cfg=json.load(open(config_path,encoding="utf-8"))
frame_start=1
frame_end=int(cfg.get("frames",120))
fps=int(cfg.get("fps",24))
video_name=pathlib.Path(cfg.get("videoFile","out/final.mp4")).name
drive_project=cfg.get("driveProjectDir",PROJECT_SLUG)

persistent_root=DRIVE_ROOT/drive_project/"resumable-render"
frames_dir=persistent_root/"frames"
logs_dir=persistent_root/"logs"
persistent_root.mkdir(parents=True,exist_ok=True)
frames_dir.mkdir(parents=True,exist_ok=True)
logs_dir.mkdir(parents=True,exist_ok=True)

progress_path=persistent_root/"render-progress.json"
final_path=persistent_root/video_name
blend_path=project_dir/"out"/pathlib.Path(cfg.get("blendFile","out/scene.blend")).name

print("Proyecto:",PROJECT_SLUG)
print("Frames:",frame_start,"-",frame_end)
print("Bloque:",BLOCK_SIZE)
print("Persistencia:",persistent_root)

section("5/8 - Construyendo escena .blend sin render completo")
(project_dir/"out").mkdir(parents=True,exist_ok=True)

env=os.environ.copy()
env["BLENDER_BUILD_ONLY"]="1"
env["BLENDER_SKIP_POSTER"]="1"

build_log=logs_dir/"build-scene.log"
with open(build_log,"w",encoding="utf-8") as log:
    p=subprocess.run(
        [BLENDER_BIN,"-b","--python",str(scene_script)],
        cwd="/content",
        env=env,
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
    )

if p.returncode!=0 or not blend_path.exists():
    tail="\n".join(build_log.read_text(errors="ignore").splitlines()[-120:])
    print(tail)
    raise RuntimeError("No se pudo construir el .blend base.")

print("Blend listo:",blend_path)

section("6/8 - Render reanudable por bloques")
def expected_frame(frame):
    return frames_dir/f"{frame:04d}.png"

def frame_done(frame):
    p=expected_frame(frame)
    return p.exists() and p.stat().st_size>1000

def completed_frames():
    return [f for f in range(frame_start,frame_end+1) if frame_done(f)]

def write_progress(status, current_block=None):
    done=completed_frames()
    payload={
        "version_marker":VERSION_MARKER,
        "project":PROJECT_SLUG,
        "status":status,
        "frame_start":frame_start,
        "frame_end":frame_end,
        "block_size":BLOCK_SIZE,
        "completed_frames":len(done),
        "total_frames":frame_end-frame_start+1,
        "percent":round(len(done)/(frame_end-frame_start+1)*100,1),
        "last_completed_frame":max(done) if done else None,
        "current_block":current_block,
        "updated_at":datetime.now(timezone.utc).isoformat(),
    }
    tmp=progress_path.with_suffix(".tmp")
    tmp.write_text(json.dumps(payload,indent=2),encoding="utf-8")
    tmp.replace(progress_path)
    print(json.dumps(payload,ensure_ascii=False))

write_progress("starting")

blocks=[]
s=frame_start
while s<=frame_end:
    e=min(frame_end,s+BLOCK_SIZE-1)
    blocks.append((s,e))
    s=e+1

for start,end in blocks:
    missing=[f for f in range(start,end+1) if not frame_done(f)]
    if not missing:
        print(f"Bloque {start}-{end}: ya existe, se omite.")
        continue

    # Para mantener continuidad y simplicidad, si un bloque esta incompleto
    # se vuelve a renderizar el bloque completo. Solo se arriesgan BLOCK_SIZE frames.
    print(f"Renderizando bloque {start}-{end}...")
    write_progress("rendering",{"start":start,"end":end})

    local_frames=pathlib.Path("/content/block-frames")
    if local_frames.exists():
        shutil.rmtree(local_frames)
    local_frames.mkdir(parents=True)

    prefix=str(local_frames/"####")

    log_path=logs_dir/f"block-{start:04d}-{end:04d}.log"
    cmd=[
        BLENDER_BIN,
        "-b",
        str(blend_path),
        "-s",str(start),
        "-e",str(end),
        "-o",prefix,
        "-F","PNG",
        "-a",
    ]

    started=time.time()
    with open(log_path,"w",encoding="utf-8") as log:
        proc=subprocess.run(
            cmd,
            cwd="/content",
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True,
        )
    elapsed=time.time()-started

    if proc.returncode!=0:
        tail="\n".join(log_path.read_text(errors="ignore").splitlines()[-100:])
        print(tail)
        write_progress("failed",{"start":start,"end":end})
        raise RuntimeError(f"Fallo el bloque {start}-{end}")

    produced=sorted(local_frames.glob("*.png"))
    if not produced:
        raise RuntimeError(f"El bloque {start}-{end} no produjo PNGs.")

    for src in produced:
        # Blender con #### produce nombres numericos; tomamos el numero final.
        digits="".join(ch for ch in src.stem if ch.isdigit())
        if not digits:
            continue
        frame=int(digits[-4:])
        if frame_start<=frame<=frame_end:
            shutil.copy2(src,expected_frame(frame))

    print(f"Bloque {start}-{end} listo en {elapsed/60:.2f} min.")
    write_progress("rendering")

done=completed_frames()
if len(done)!=(frame_end-frame_start+1):
    missing=[f for f in range(frame_start,frame_end+1) if not frame_done(f)]
    raise RuntimeError(f"Faltan frames: {missing[:20]}")

write_progress("frames_complete")

section("7/8 - Ensamblando video final")
# Secuencia PNG -> MP4. El pad/format yuv420p maximizan compatibilidad móvil.
ffmpeg_cmd=[
    "ffmpeg","-y",
    "-framerate",str(fps),
    "-start_number",str(frame_start),
    "-i",str(frames_dir/"%04d.png"),
    "-c:v","libx264",
    "-preset","medium",
    "-crf","18",
    "-pix_fmt","yuv420p",
    str(final_path),
]
q=subprocess.run(ffmpeg_cmd,cwd="/content")
if q.returncode!=0 or not final_path.exists():
    raise RuntimeError("FFmpeg no pudo ensamblar el video.")

write_progress("completed")

section("8/8 - LISTO")
print("Video:",final_path)
print("Tamano:",round(final_path.stat().st_size/1024/1024,2),"MB")
print("Frames persistentes:",len(completed_frames()))
print("Progreso:",progress_path)
print("Si Colab se cae, vuelve a Ejecutar todo: los bloques terminados se omiten.")
